In [ ]:
!pip install ultralytics -q
print("Ultralytics is installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.7 MB/s eta 0:00:00a 0:00:01
✓ Ultralytics is installed

In [ ]:
import torch
assert torch.cuda.is_available(),
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

GPU: Tesla T4

VRAM: 15.6 GB

In [ ]:
from pathlib import Path

DATASET_DIR = "/kaggle/input/datasets/natair/chicken-dataset/chicken_dataset"

WORK_DIR    = "/kaggle/working"
RUN_NAME    = "chicken_detector"
IMG_SIZE    = 1280
TOTAL_EPOCHS = 150
BATCH       = 8
MODEL_BASE  = "yolov8m.pt"

In [ ]:
# Creating a dataset config for YOLO

data_yaml = f"""
path: {DATASET_DIR}
train: images/train
val: images/val

names:
  0: chicken
"""

yaml_path = f"{WORK_DIR}/data.yaml"
with open(yaml_path, "w") as f:
    f.write(data_yaml)

print(f"data.yaml created: {yaml_path}")
print(data_yaml)

train_imgs = list(Path(DATASET_DIR, "images/train").glob("*.jpg"))
val_imgs   = list(Path(DATASET_DIR, "images/val").glob("*.jpg"))
train_lbls = list(Path(DATASET_DIR, "labels/train").glob("*.txt"))
val_lbls   = list(Path(DATASET_DIR, "labels/val").glob("*.txt"))

print(f"Train: {len(train_imgs)} images / {len(train_lbls)} annotaions")
print(f"Val:   {len(val_imgs)} images / {len(val_lbls)} annotaions")

assert len(train_imgs) > 0, "No train images found."
assert len(train_imgs) == len(train_lbls), "The number of images and annotations does not match!"

data.yaml created: /kaggle/working/data.yaml

path: /kaggle/input/datasets/natair/chicken-dataset/chicken_dataset
train: images/train
val: images/val

names:
0: chicken

Train: 80 images / 80 annotations
Val:   19 images / 19 annotations

In [ ]:
from ultralytics import YOLO
import os

RESUME_CHECKPOINT = None
# "/kaggle/input/chicken-checkpoint/last.pt"

if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).exists():
    print(f"Continuing training from the checkpoint: {RESUME_CHECKPOINT}")
    model = YOLO(RESUME_CHECKPOINT)
    resume_flag = True
else:
    print(f"Start training from scratch: {MODEL_BASE}")
    model = YOLO(MODEL_BASE)
    resume_flag = False

results = model.train(
    data=yaml_path,
    epochs=TOTAL_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name=RUN_NAME,
    project=f"{WORK_DIR}/runs",
    resume=resume_flag,

    # Augmentations for small objects
    mosaic=1.0,
    scale=0.9,
    copy_paste=0.3,
    degrees=10.0,
    fliplr=0.5,
    flipud=0.3,
    hsv_h=0.015,
    hsv_v=0.4,

    # Session interruption resilience
    save=True,
    save_period=5,      # checkpoint every 5 epochs
    patience=40,        # early discontinuation if there is no improvement over a prolonged period
    exist_ok=True,      # write to the same 'run' folder again
    verbose=True,
    plots=True,
)

print("\n Training completed (or interrupted due to patience)")
print(f"  best.pt: {WORK_DIR}/runs/{RUN_NAME}/weights/best.pt")
print(f"  last.pt: {WORK_DIR}/runs/{RUN_NAME}/weights/last.pt")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
▶ Начинаем обучение с нуля: yolov8m.pt
Downloading https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8m.pt to 'yolov8m.pt': 100% ━━━━━━━━━━━━ 49.7MB 176.5MB/s 0.3s0.2s<0.2s
Ultralytics 8.4.87  Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=chicken_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=40, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=/kaggle/working/runs, quantize=None, rect=False, resume=False, retina_masks=False, rle=1.0, save=True, save_conf=False, save_crop=False, save_dir=/kaggle/working/runs/chicken_detector, save_frames=False, save_json=False, save_period=5, save_txt=False, scale=0.9, seed=0, shear=0.0, show=False, show_boxes=True, show_conf=True, show_labels=True, simplify=True, single_cls=False, source=None, split=val, stream_buffer=False, task=detect, time=None, tracker=tracktrack.yaml, translate=0.1, val=True, verbose=True, vid_stride=1, visualize=False, warmup_bias_lr=0.1, warmup_epochs=3.0, warmup_momentum=0.8, weight_decay=0.0005, workers=8, workspace=None
Downloading https://ultralytics.com/assets/Arial.ttf to '/root/.config/Ultralytics/Arial.ttf': 100% ━━━━━━━━━━━━ 755.1KB 16.6MB/s 0.0s
Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics.nn.modules.conv.Conv             [384, 576, 3, 2]              
  8                  -1  2   3985920  ultralytics.nn.modules.block.C2f             [576, 576, 2, True]           
  9                  -1  1    831168  ultralytics.nn.modules.block.SPPF            [576, 576, 5]                 
 10                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 11             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 12                  -1  2   1993728  ultralytics.nn.modules.block.C2f             [960, 384, 2]                 
 13                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 14             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 15                  -1  2    517632  ultralytics.nn.modules.block.C2f             [576, 192, 2]                 
 16                  -1  1    332160  ultralytics.nn.modules.conv.Conv             [192, 192, 3, 2]              
 17            [-1, 12]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 18                  -1  2   1846272  ultralytics.nn.modules.block.C2f             [576, 384, 2]                 
 19                  -1  1   1327872  ultralytics.nn.modules.conv.Conv             [384, 384, 3, 2]              
 20             [-1, 9]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 21                  -1  2   4207104  ultralytics.nn.modules.block.C2f             [960, 576, 2]                 
 22        [15, 18, 21]  1   3776275  ultralytics.nn.modules.head.Detect           [1, 16, None, [192, 384, 576]]
Model summary: 170 layers, 25,856,899 parameters, 25,856,883 gradients, 79.1 GFLOPs

Transferred 469/475 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
Downloading https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n.pt to 'yolo26n.pt': 100% ━━━━━━━━━━━━ 5.3MB 68.8MB/s 0.1s
AMP: checks passed 
train: Fast image access  (ping: 0.2±0.3 ms, read: 63.5±46.6 MB/s, size: 1206.8 KB)
train: Scanning /kaggle/input/datasets/natair/chicken-dataset/chicken_dataset/labels/train... 80 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 182.1it/s 0.4s0.2s
WARNING  train: Cache directory /kaggle/input/datasets/natair/chicken-dataset/chicken_dataset/labels is not writable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access (ping: 0.6±0.3 ms, read: 94.0±68.1 MB/s, size: 1232.4 KB)
val: Scanning /kaggle/input/datasets/natair/chicken-dataset/chicken_dataset/labels/val... 19 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 19/19 223.0it/s 0.1s
WARNING val: Cache directory /kaggle/input/datasets/natair/chicken-dataset/chicken_dataset/labels is not writable, cache not saved.
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
Plotting labels to /kaggle/working/runs/chicken_detector/labels.jpg... 
Image sizes 1280 train, 1280 val
Using 2 dataloader workers
Logging results to /kaggle/working/runs/chicken_detector
Starting training for 150 epochs...

Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

WARNING CUDA out of memory with batch=16. Reducing to batch=8 and retrying (1/3).train: Fast image access (ping: 0.0±0.0 ms, read: 1349.0±363.6 MB/s, size: 1243.7 KB)train: Scanning /kaggle/input/datasets/natair/chicken-dataset/chicken_dataset/labels/train... 80 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 727.9it/s 0.1s0.0s

WARNING train: Cache directory /kaggle/input/datasets/natair/chicken-dataset/chicken_dataset/labels is not writable, cache not saved.albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))val: Fast image access (ping: 0.0±0.0 ms, read: 545.3±250.4 MB/s, size: 1118.7 KB)val: Scanning /kaggle/input/datasets/natair/chicken-dataset/chicken_dataset/labels/val... 19 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 19/19 528.8it/s 0.0s

WARNING val: Cache directory /kaggle/input/datasets/natair/chicken-dataset/chicken_dataset/labels is not writable, cache not saved.optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)

: 0% ──────────── 0/5 1.6s



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

1/150 12.1G 3.456 6.072 1.841 217 1280: 100% ━━━━━━━━━━━━ 10/10 1.3s/it 12.6s.1s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4s/it 1.4s

all 19 280 0.342 0.311 0.201 0.0561



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

2/150 11.8G 2.652 5.457 1.396 162 1280: 100% ━━━━━━━━━━━━ 10/10 1.0it/s 9.8s.0ss

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.6it/s 0.6s

all 19 280 0.0424 0.0321 0.0119 0.00536



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

3/150 11.7G 2.217 2.609 1.218 85 1280: 100% ━━━━━━━━━━━━ 10/10 1.0it/s 9.8s.0ss

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.6it/s 0.6s

all 19 280 0.292 0.244 0.111 0.0381



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

4/150 11.9G 2.189 2.509 1.259 105 1280: 100% ━━━━━━━━━━━━ 10/10 1.0s/it 10.0s.0s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.6it/s 0.6s

all 19 280 0.385 0.434 0.27 0.0889



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

5/150 12G 2.219 1.98 1.229 113 1280: 100% ━━━━━━━━━━━━ 10/10 1.0s/it 10.2s.1s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.5it/s 0.7s

all 19 280 0.396 0.293 0.224 0.0749



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

6/150 12.2G 2.146 2.05 1.222 263 1280: 100% ━━━━━━━━━━━━ 10/10 1.0s/it 10.4s.1s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.6it/s 0.6s

all 19 280 0.124 0.0964 0.0421 0.0173



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

7/150 11.7G 2.22 1.854 1.337 97 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 10.6s.1s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.6it/s 0.6s

all 19 280 0.489 0.0107 0.015 0.006



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

8/150 11.9G 2.222 1.87 1.216 246 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.0s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.1it/s 0.9s

all 19 280 0 0 0 0



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

9/150 12.2G 2.422 2.055 1.32 62 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.1it/s 0.9s

all 19 280 0 0 0 0



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

10/150 11.7G 2.222 1.629 1.227 200 1280: 100% ━━━━━━━━━━━━ 10/10 1.2s/it 11.9s.3s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0 0 0 0



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

11/150 12.4G 2.355 1.756 1.26 179 1280: 100% ━━━━━━━━━━━━ 10/10 1.2s/it 12.3s.3s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0 0 0 0



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

12/150 12.3G 2.262 1.555 1.366 182 1280: 100% ━━━━━━━━━━━━ 10/10 1.2s/it 11.6s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.000175 0.00357 1.41e-06 1.41e-07



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

13/150 12G 2.378 1.876 1.367 115 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.3s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.1it/s 0.9s

all 19 280 0 0 0 0



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

14/150 12.1G 2.266 1.803 1.296 138 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.3s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.2it/s 0.8s

all 19 280 0 0 0 0



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

15/150 11.6G 2.203 1.575 1.255 86 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.3s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.2it/s 0.8s

all 19 280 0.00105 0.0214 2.65e-05 1.21e-05



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

16/150 11.9G 2.132 1.403 1.162 113 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.35 0.296 0.216 0.08



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

17/150 11.7G 2.276 1.504 1.237 96 1280: 100% ━━━━━━━━━━━━ 10/10 1.2s/it 11.6s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.39 0.354 0.277 0.1



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

18/150 12G 2.012 1.363 1.265 92 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.308 0.146 0.114 0.0499



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

19/150 12.2G 2.115 1.425 1.228 66 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.5it/s 0.7s

all 19 280 0.299 0.125 0.0909 0.0372



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

20/150 12G 2.133 1.404 1.211 141 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.5it/s 0.7s

all 19 280 0.314 0.125 0.1 0.0391



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

21/150 12.3G 2.044 1.319 1.13 114 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.5it/s 0.7s

all 19 280 0.274 0.075 0.0511 0.02



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

22/150 12.3G 2.067 1.289 1.231 181 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.393 0.271 0.241 0.103



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

23/150 11.8G 2.079 1.258 1.191 139 1280: 100% ━━━━━━━━━━━━ 10/10 1.2s/it 11.6s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.42 0.354 0.323 0.131



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

24/150 12.1G 2.027 1.208 1.168 149 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.45 0.464 0.383 0.14



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

25/150 11.7G 2.129 1.38 1.167 59 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.518 0.454 0.411 0.133



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

26/150 11.7G 1.99 1.29 1.172 267 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.567 0.571 0.513 0.174



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

27/150 11.9G 2.064 1.273 1.208 118 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.537 0.547 0.466 0.16



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

28/150 12G 2.041 1.307 1.188 179 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.58 0.454 0.466 0.165



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

29/150 12.6G 2.05 1.236 1.148 133 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.441 0.291 0.301 0.12



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

30/150 12.4G 2.072 1.258 1.19 134 1280: 100% ━━━━━━━━━━━━ 10/10 1.2s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.466 0.396 0.354 0.14



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

31/150 12.2G 2.097 1.284 1.174 154 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.0456 0.261 0.0139 0.00541



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

32/150 11.9G 2.029 1.327 1.173 90 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.039 0.236 0.0105 0.00421



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

33/150 12G 2.124 1.315 1.097 222 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.515 0.496 0.49 0.204



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

34/150 12G 1.988 1.244 1.161 68 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.627 0.514 0.58 0.225



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

35/150 11.9G 2.072 1.272 1.146 87 1280: 100% ━━━━━━━━━━━━ 10/10 1.2s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.57 0.654 0.632 0.247



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

36/150 12.4G 2.018 1.2 1.107 252 1280: 100% ━━━━━━━━━━━━ 10/10 1.2s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.624 0.646 0.635 0.25



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

37/150 12G 2.093 1.272 1.103 168 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.526 0.643 0.575 0.228



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

38/150 12.2G 2.012 1.179 1.152 306 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.649 0.615 0.597 0.205



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

39/150 11.6G 2.058 1.239 1.157 122 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.5it/s 0.7s

all 19 280 0.66 0.597 0.561 0.199



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

40/150 11.7G 2.046 1.2 1.16 157 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.676 0.6 0.585 0.232



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

41/150 11.9G 1.974 1.226 1.133 195 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.3s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.664 0.571 0.607 0.236



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

42/150 11.9G 1.984 1.278 1.178 142 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.5it/s 0.7s

all 19 280 0.587 0.589 0.585 0.233



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

43/150 11.9G 1.972 1.164 1.175 119 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.609 0.586 0.596 0.248



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

44/150 11.9G 1.874 1.139 1.149 204 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.609 0.623 0.603 0.252



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

45/150 12.1G 2.053 1.172 1.128 183 1280: 100% ━━━━━━━━━━━━ 10/10 1.2s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.661 0.618 0.645 0.259



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

46/150 11.6G 1.934 1.101 1.134 89 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.614 0.625 0.606 0.232



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

47/150 11.8G 1.918 1.154 1.183 68 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.616 0.611 0.58 0.228



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

48/150 12.1G 1.95 1.119 1.112 157 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.62 0.593 0.592 0.234



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

49/150 11.6G 1.863 1.095 1.163 171 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.4s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.625 0.625 0.621 0.237



Epoch GPU_mem box_loss cls_loss dfl_loss Instances Size

50/150 11.6G 1.927 1.161 1.122 146 1280: 100% ━━━━━━━━━━━━ 10/10 1.1s/it 11.5s.2s

Class Images Instances Box(P R mAP50 mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.4it/s 0.7s

all 19 280 0.624 0.632 0.606 0.237






In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_csv = f"{WORK_DIR}/runs/{RUN_NAME}/results.csv"
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

print("Last 5 epochs:")
print(df[["epoch", "metrics/precision(B)", "metrics/recall(B)",
          "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50")
axes[0].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
axes[0].set_title("mAP by epochs")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df["epoch"], df["metrics/precision(B)"], label="Precision")
axes[1].plot(df["epoch"], df["metrics/recall(B)"], label="Recall")
axes[1].set_title("Precision / Recall")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{WORK_DIR}/training_summary.jpg", dpi=120)
plt.show()

best_map50 = df["metrics/mAP50(B)"].max()
print(f"\nBest mAP50: {best_map50:.3f}")

Last 5 epochs:
     epoch  metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  \
145    146               0.67074            0.67661           0.66906   
146    147               0.69181            0.66786           0.67785   
147    148               0.70850            0.65103           0.68994   
148    149               0.70206            0.67325           0.69002   
149    150               0.69507            0.67143           0.68828   

     metrics/mAP50-95(B)  
145              0.26270  
146              0.27070  
147              0.27773  
148              0.27638  
149              0.27925  

![](data/images/Screenshot%202026-09-15%20at%2015.44.15.png)

Best mAP50: 0.732
A good result for a first model.

In [ ]:
best_model = YOLO(f"{WORK_DIR}/runs/{RUN_NAME}/weights/best.pt")

# Pick any image from 'val' for a visual check.
test_img = str(val_imgs[0])
res = best_model(test_img, imgsz=IMG_SIZE, conf=0.25)

res[0].save(filename=f"{WORK_DIR}/test_prediction.jpg")
print(f"Result saved: {WORK_DIR}/test_prediction.jpg")
print(f"Objects found: {len(res[0].boxes)}")

import cv2
img = cv2.cvtColor(cv2.imread(f"{WORK_DIR}/test_prediction.jpg"), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 9))
plt.imshow(img)
plt.axis("off")
plt.title(f"Detections: {len(res[0].boxes)}")
plt.show()

image 1/1 /kaggle/input/datasets/natair/chicken-dataset/chicken_dataset/images/val/frame_023664.jpg: 736x1280 2 chickens, 54.1ms
Speed: 6.1ms preprocess, 54.1ms inference, 1.2ms postprocess per image at shape (1, 3, 736, 1280)
Result saved: /kaggle/working/test_prediction.jpg
Objects found: 2

![](data/images/Screenshot%202026-09-15%20at%2015.43.52.png)